# 03 Modeling and Segmentation
Build an explainable churn score and prepare RFM features for K-Means.

In [1]:
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

DATA_PATH = '../data/processed/master_cleaned_data.csv'
data = pd.read_csv(DATA_PATH, parse_dates=['Order_Date'])
reference = data['Order_Date'].max()

# Customer-level RFM features and transparent churn-risk summary.
rfm = data.groupby('Customer_ID').agg(
    Recency=('Order_Date', lambda values: (reference - values.max()).days),
    Frequency=('Order_Date', 'count'),
    Monetary=('Total_Revenue', 'sum'),
    Churn=('Churn', 'max'),
    Churn_Risk_Score=('Churn_Risk_Score', 'max'),
)
features = StandardScaler().fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])
cluster_count = min(3, len(rfm))
rfm['Segment'] = KMeans(n_clusters=cluster_count, random_state=42, n_init=10).fit_predict(features)

segment_summary = rfm.groupby('Segment').agg(
    Customers=('Monetary', 'size'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Revenue=('Monetary', 'sum'),
    Avg_Risk=('Churn_Risk_Score', 'mean'),
).sort_values('Revenue', ascending=False)

churned = data.loc[data['Churn'].eq(1), 'Total_Revenue']
retained = data.loc[data['Churn'].eq(0), 'Total_Revenue']
welch_test = stats.ttest_ind(churned, retained, equal_var=False)

print('Customer segments:')
print(segment_summary.round(2))
print(f'High-risk customers: {(rfm.Churn_Risk_Score >= 60).sum()}')
print(f"Welch t-test p-value (churned vs retained revenue): {welch_test.pvalue:.4f}")
rfm.sort_values('Monetary', ascending=False).head()

Customer segments:
         Customers  Avg_Recency  Avg_Frequency  Revenue  Avg_Risk
Segment                                                          
2                2          0.5            2.0  1547.98     22.90
1                2          2.0            1.0   839.15     70.40
0                2          0.5            1.0   363.00     49.15
High-risk customers: 2
Welch t-test p-value (churned vs retained revenue): 0.0538


c:\Users\prajy\.conda\envs\myenv\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,Recency,Frequency,Monetary,Churn,Churn_Risk_Score,Segment
Customer_ID,,,,,,
C001,1,2,1061.98,0,27.9,2
C004,2,1,764.15,0,55.4,1
C002,0,2,486.00,0,17.9,2
C006,0,1,249.00,0,30.4,0
C005,1,1,114.00,1,67.9,0
